In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import os
import json

from torch import from_numpy, Tensor, where

from pathlib import WindowsPath

from skimage.color import rgb2gray
from skimage.io import imread, imsave
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
# from skimage.restoration import richardson_lucy

from olimp.precompensation.nn.dataset import psf_gauss
from olimp.precompensation.basic.huang import huang
from olimp.processing import fft_conv

from typing import Dict, Tuple, Optional

# Генерация датасета из 36 изображений

In [ ]:
# task initialization


MAX_UINT8_INTENSITY = 255

ORIG_DIRPATH = WindowsPath(r"data\orig_images")
DEST_DIRPATH = WindowsPath(r"data\processed")

IMAGE_SIZE = 512
PSF_TUPLE = (
    psf_gauss.PsfGaussDataset(
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        center_x=0, center_y=0,
        theta=0,
        sigma_x=10, sigma_y=10,
        size=1
    ),
    psf_gauss.PsfGaussDataset(
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        center_x=0, center_y=0,
        theta=0,
        sigma_x=5, sigma_y=5,
        size=1
    ),
    psf_gauss.PsfGaussDataset(
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        center_x=0, center_y=0,
        theta=0,
        sigma_x=10, sigma_y=5,
        size=1
    )
)

STANDART_DEVIATIONS = (0.01, 0.05, 0.1)  # for noise

np.random.seed(42)

In [ ]:
def preprocess_image(
    image: np.ndarray,
) -> np.ndarray:
    """
    Transforms colored image to gray and changes dtype from uint8 [0, 255] to float32 [0, 1].
    """
    image_float = image.astype(np.float32) / MAX_UINT8_INTENSITY
    gray = rgb2gray(image_float)
    return gray


def load_images(
    orig_dirpath: WindowsPath
) -> Dict[str, np.ndarray]:
    """
    Loads and preprocesses original images.
    """
    images = dict()

    for filename in os.listdir(orig_dirpath):
        images[filename] = preprocess_image(imread(orig_dirpath.joinpath(os.path.basename(filename))))
    
    return images


def add_gaussian_noise(
    image: np.ndarray,
    std: float
) -> np.ndarray:
    """
    Adds Gaussian noise N(0, std^2) to image.
    """
    blurred_image = image + np.random.normal(0, std, image.shape)
    blurred_image.clip(0, 1)
    return blurred_image


def process_triplet(
    image: np.ndarray,
    psf: Tensor,
    std: float
) -> Dict[str, np.ndarray]:
    """
    Processes single image and returns Dict of original, convoluted & blurred images.
    """
    convoluted_image = fft_conv(from_numpy(image), psf).numpy()

    return {
        "orig.png": image,
        "conv.png": convoluted_image,
        "blur.png": add_gaussian_noise(convoluted_image, std)
    }


def save_triplet(
    triplet: Dict[str, np.ndarray],
    dest: WindowsPath
) -> None:
    """
    Saves single image triplet on dest.
    """
    for name in triplet.keys():
        imsave(dest.joinpath(name), np.uint8(triplet[name] * MAX_UINT8_INTENSITY))


def save_processed_images(
    images: Dict[str, np.ndarray],
    psf_tuple: Tuple[psf_gauss.PsfGaussDataset],
    std_tuple: Tuple[float, float, float],
    dest_dirpath: WindowsPath
) -> None:
    """
    Processes & saves images.
    """
    for name in images.keys():
        for i, psf in enumerate(psf_tuple):
            for std in std_tuple:
                dest = dest_dirpath.joinpath(f"img_{name}.psf_{i}.std_{std:.2f}")
                dest.mkdir()
                save_triplet(process_triplet(images[name], psf[0], std), dest)

def generate_dataset(
    orig_dirpath: WindowsPath,
    dest_dirpath: WindowsPath,
    psf_tuple: Tuple[psf_gauss.PsfGaussDataset],
    std_tuple: Tuple[float, float, float],
) -> None:
    """
    Generates dataset.
    """
    images = load_images(orig_dirpath)
    dest_dirpath.mkdir()
    save_processed_images(images, psf_tuple, std_tuple, dest_dirpath)



In [ ]:
generate_dataset(
    ORIG_DIRPATH,
    DEST_DIRPATH,
    PSF_TUPLE,  # type: ignore
    STANDART_DEVIATIONS
)

In [ ]:
print(f"dataset size: {len(os.listdir(DEST_DIRPATH))}")

fig, axes = plt.subplots(1, 3, figsize=(9, 27))

axes[0].imshow(imread(r"data\processed\img_4.2.03.tiff.psf_0.std_0.01\orig.png"), cmap='gray')
axes[1].imshow(imread(r"data\processed\img_4.2.03.tiff.psf_0.std_0.01\conv.png"), cmap='gray')
axes[2].imshow(imread(r"data\processed\img_4.2.03.tiff.psf_0.std_0.01\blur.png"), cmap='gray')

axes[0].set_title("orig")
axes[1].set_title("conv")
axes[2].set_title("noise")

plt.show()

# Деконволюция изображений

In [ ]:
def richardson_lucy(
    image: Tensor,
    psf: Tensor,
    num_iter=50,
):
    image = image
    im_deconv = from_numpy(np.full(image.shape, 0.5, np.float32))
    psf_mirror = psf.flip([0, 1, 2])

    # Small regularization parameter used to avoid 0 divisions
    eps = 1e-12
    filter_epsilon = 1e-7

    for _ in range(num_iter):
        conv = fft_conv(im_deconv, psf) + eps
        relative_blur = where(conv < filter_epsilon, 0, image / conv)
        im_deconv *= fft_conv(relative_blur, psf_mirror)[0]

    return im_deconv.numpy().clip(0, 1)


def deconvolute_dir(
    dest_dirpath: WindowsPath,
    psf: psf_gauss.PsfGaussDataset
) -> Dict[str, Dict[str, float]]:
    orig_img = imread(dest_dirpath.joinpath("orig.png")).astype(np.float32) / MAX_UINT8_INTENSITY
    blur_img = imread(dest_dirpath.joinpath("blur.png")).astype(np.float32) / MAX_UINT8_INTENSITY

    metrix = dict()

    # blur
    metrix["blur"] = {
        "psnr": psnr(orig_img, blur_img, data_range=1.0),
        "ssim": ssim(orig_img, blur_img, data_range=1.0)
    }

    # wiener
    wiener = huang(from_numpy(blur_img), psf[0]).numpy()[0]
    metrix["wiener"] = {
        "psnr": psnr(orig_img, wiener, data_range=1.0),
        "ssim": ssim(orig_img, wiener, data_range=1.0)
    }

    imsave(dest_dirpath.joinpath("wiener.png"), (wiener * MAX_UINT8_INTENSITY).astype(np.uint8))

    # RL
    rl = richardson_lucy(blur_img, psf[0])
    metrix["rl"] = {
        "psnr": psnr(orig_img, rl, data_range=1.0),
        "ssim": ssim(orig_img, rl, data_range=1.0)
    }
    imsave(dest_dirpath.joinpath("rl.png"), (rl * MAX_UINT8_INTENSITY).astype(np.uint8))

    with open(dest_dirpath.joinpath("metrix.json"), 'w') as file:
        json.dump(metrix, file)

    return metrix


def get_psf_index(
    dirname: str
) -> int:
    return int(dirname[20])

def deconvolute_dataset(
    dataset_dirpath: WindowsPath,
    psf_tuple: Tuple[psf_gauss.PsfGaussDataset]
) -> None:
    metrix = {
        "blur": {"psnr": 0.0, "ssim": 0.0},
        "wiener": {"psnr": 0.0, "ssim": 0.0},
        "rl": {"psnr": 0.0, "ssim": 0.0}
    }

    size = len(os.listdir(dataset_dirpath))

    for dirname in os.listdir(dataset_dirpath):
        index = get_psf_index(dirname)
        mnew = deconvolute_dir(dataset_dirpath.joinpath(dirname), psf_tuple[index])

        metrix["blur"]["psnr"] += mnew["blur"]["psnr"] / size
        metrix["blur"]["ssim"] += mnew["blur"]["ssim"] / size

        metrix["wiener"]["psnr"] += mnew["wiener"]["psnr"] / size
        metrix["wiener"]["ssim"] += mnew["wiener"]["ssim"] / size

        metrix["rl"]["psnr"] += mnew["rl"]["psnr"] / size
        metrix["rl"]["ssim"] += mnew["rl"]["ssim"] / size
    
    with open(dataset_dirpath.joinpath("metrix_mean.json"), 'w') as file:
        json.dump(metrix, file)



In [ ]:
deconvolute_dataset(
    DEST_DIRPATH,
    PSF_TUPLE  # type: ignore
)